# ECOSIM-G en Kaggle: 1.000 economías a la vez en las dos T4

**ECOSIM-G** es el Monte Carlo de economías cerradas escrito en Ñ para **C1EÑ**: cada economía es un «mundo» con sus compradores (hambre, aburrimiento, curiosidad, colchón), sus empresas (producen, pagan nómina, invierten en máquinas, quiebran), sus inversores (valor / tendencia / azar, euforia y pánico, compras a margen), sus **bancos** (doble entrada, encaje del 10 %, llamadas de margen, castigos, crisis de liquidez) y sus **clubes** de compañeros de trabajo (voluntad, participaciones, corridas, fundan empresas). El dinero (depósitos − préstamos = base), las deudas, las acciones, los bienes y las participaciones se verifican cada minuto simulado, en la GPU.

Cómo entra en la GPU: todo el estado vive en arreglos planos de tamaño `economías × N`; el hilo `h` es el agente `h % N` de la economía `h // N`, y cada regla es un `paralelo` sobre todas las economías a la vez. Los arreglos se quedan **residentes** en la GPU entre núcleos: por el PCIe solo pasa lo que el programa lee (el CSV cada 10 minutos). El programa es `ecosim/ecosim-g.esl`, y corre igual (al bit) en `jit`, `nativo` y `gpu`.

Este cuaderno:
1. instala Node y koffi y consigue el proyecto (dataset subido por ti **o** `git clone`);
2. hace una corrida pequeña en la GPU 0 para comprobar que todo cuadra;
3. reparte **1.000 economías entre las dos T4**: un proceso por GPU (`C1EN_GPU=0` y `C1EN_GPU=1`), 500 economías cada uno, con semillas distintas;
4. junta los dos CSV y pinta la distribución: índice de bolsa final, quiebras, fundaciones, paro, préstamos, clubes, y el abanico de los mil índices.

**Antes de empezar:** *Settings → Accelerator → GPU T4 x2*. Si vas a usar la opción B (clonar de GitHub), también *Settings → Internet → On*.

Parámetros del programa: `node motor/util/n-node.js --c1en ecosim/ecosim-g.esl [economias] [ticks] [compradores] [semilla] [salida.csv|-] [empresasPorSector] [credito]`. Un tick es un segundo simulado; 7200 ticks son 2 horas. Los arreglos enteros son de 32 bits, así que el dinero de UNA economía debe caber en 2^31 centavos (≈ 21 millones de monedas): con 10.000 compradores van 16 millones. `C1EN_ESTADISTICAS=1` imprime al salir cuántos lanzamientos y cuántos MB pasaron por el PCIe.


In [ ]:
# 1 · Node 22 (Kaggle trae uno viejo o ninguno) — se instala en /kaggle/working/node sin tocar el sistema
import os, subprocess, shutil, re
def version_node():
    try: return subprocess.run(['node', '--version'], capture_output=True, text=True).stdout.strip()
    except FileNotFoundError: return ''
v = version_node()
if not v or int(re.match(r'v(\d+)', v).group(1)) < 18:
    NODE = 'v22.12.0'
    subprocess.run(f'curl -fsSL https://nodejs.org/dist/{NODE}/node-{NODE}-linux-x64.tar.xz | tar -xJ -C /kaggle/working', shell=True, check=True)
    os.environ['PATH'] = f'/kaggle/working/node-{NODE}-linux-x64/bin:' + os.environ['PATH']
print('node', version_node())

In [ ]:
# 2 · El proyecto. Opción A: un dataset de Kaggle con la carpeta del proyecto (la que tiene motor/ dentro).
#     Opción B: git clone del repositorio. Se usa A si se encuentra; si no, B.
import glob
PROYECTO = '/kaggle/working/n'
candidatos = [os.path.dirname(os.path.dirname(p)) for p in glob.glob('/kaggle/input/**/motor/rutas.js', recursive=True)]
if os.path.exists(PROYECTO):
    print('ya estaba en', PROYECTO)
elif candidatos:
    # /kaggle/input es de solo lectura y hay que compilar el bundle e instalar koffi: se copia.
    shutil.copytree(candidatos[0], PROYECTO, ignore=shutil.ignore_patterns('node_modules'))
    print('opción A: copiado desde', candidatos[0])
else:
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/diegovelandiabarajas1-lang/Espanol-Like', PROYECTO], check=True)
    print('opción B: clonado en', PROYECTO)
os.chdir(PROYECTO)
assert os.path.exists('motor/src/14-c1en.js'), 'este proyecto no trae C1EÑ (motor/src/14-c1en.js): ¿es una versión vieja?'

In [ ]:
# 3 · koffi (el FFI de Node que habla con libcuda y libnvrtc) y el bundle del motor
!cd /kaggle/working/n && npm install koffi --no-audit --no-fund --loglevel=error && node motor/util/construir.js

In [ ]:
# 4 · Las GPU y el toolkit. C1EÑ busca libnvrtc solo; si no la encuentra, indícala con C1EN_NVRTC_LIB.
!nvidia-smi
!ls /usr/local/cuda/lib64/ 2>/dev/null | grep -E '^libnvrtc\.so' || echo 'no hay libnvrtc en /usr/local/cuda/lib64'
!cd /kaggle/working/n && node -e "require('./motor/bundle.js'); const m = globalThis.EspanolLike.crearMotor({}); console.log(m.vm.c1en.disponibles().join('\n'))"

In [ ]:
# 5 · ecosim-g.esl debe estar en el proyecto
assert os.path.exists('ecosim/ecosim-g.esl'), 'falta ecosim/ecosim-g.esl: sube la versión del proyecto que lo trae'
print(open('ecosim/ecosim-g.esl', encoding='utf-8').read()[:1200])

## Una corrida pequeña en la GPU 0

10 economías × 2 horas × 2.000 compradores. Al final el programa dice si los invariantes (dinero, acciones y bienes) se cumplieron en todas las economías y resume las crisis.

In [ ]:
import subprocess, os, time
def correr(economias, ticks, compradores, semilla, csv, gpu, por_sector=6, credito=1, log=None):
    env = dict(os.environ, C1EN_DISPOSITIVO='gpu', C1EN_GPU=str(gpu), C1EN_ESTADISTICAS='1')
    cmd = ['node', 'motor/util/n-node.js', '--c1en', 'ecosim/ecosim-g.esl', str(economias), str(ticks), str(compradores), str(semilla), csv, str(por_sector), str(credito)]
    return subprocess.Popen(cmd, env=env, stdout=open(log, 'w') if log else None, stderr=subprocess.STDOUT, text=True)

t0 = time.time()
p = correr(10, 7200, 2000, 1, '/kaggle/working/eco-prueba.csv', gpu=0, log='/kaggle/working/eco-prueba.log')
p.wait()
print(open('/kaggle/working/eco-prueba.log').read())
print(f'{time.time() - t0:.1f} s en total (incluye compilar los núcleos con NVRTC la primera vez)')

## Las 1.000 economías, 500 por GPU

Dos procesos a la vez, cada uno con su GPU. La economía `k` usa la semilla `SEMILLA + k`, así que el segundo proceso arranca en `SEMILLA + 500` para que ningún mundo se repita. Cambia `TICKS` para simular más horas (28800 = 8 h) y `COMPRADORES` para economías más grandes.

In [ ]:
ECONOMIAS = 1000          # en total
TICKS = 7200              # 2 horas simuladas (28800 = 8 h)
COMPRADORES = 2000        # por economía
SEMILLA = 20261013
POR_SECTOR = 6            # 6 × 5 sectores = 30 empresas al empezar, hasta 48
CREDITO = 1               # 0 para correr los mismos mundos sin bancos que presten

mitad = ECONOMIAS // 2
t0 = time.time()
procs = [
    correr(mitad, TICKS, COMPRADORES, SEMILLA, '/kaggle/working/eco-gpu0.csv', gpu=0, por_sector=POR_SECTOR, credito=CREDITO, log='/kaggle/working/eco-gpu0.log'),
    correr(ECONOMIAS - mitad, TICKS, COMPRADORES, SEMILLA + mitad, '/kaggle/working/eco-gpu1.csv', gpu=1, por_sector=POR_SECTOR, credito=CREDITO, log='/kaggle/working/eco-gpu1.log'),
]
for p in procs: p.wait()
print(f'{time.time() - t0:.1f} s de pared para las {ECONOMIAS} economías × {TICKS} ticks')
for i in (0, 1):
    print(f'\n── GPU {i}')
    print(open(f'/kaggle/working/eco-gpu{i}.log').read())

## Juntar los dos CSV y mirar la distribución

Cada fila es, cada 10 minutos simulados y por economía: `tick, economia, paro, indiceBolsa, activas, quiebras, fundadas, fundadasClub, producido, consumido, prestamosVivos, clubs, socios, llamadasMargen, castigos, iliquidez, corridas`. El índice está en centavos (10000 = 100 monedas, el precio de salida); los préstamos y castigos, en centavos.

In [ ]:
import pandas as pd, numpy as np
a = pd.read_csv('/kaggle/working/eco-gpu0.csv')
b = pd.read_csv('/kaggle/working/eco-gpu1.csv'); b['economia'] += mitad
df = pd.concat([a, b], ignore_index=True)
df['indice'] = df['indiceBolsa'] / 100
df.to_csv('/kaggle/working/ecosim-g-1000.csv', index=False)
final = df[df['tick'] == df['tick'].max()].set_index('economia')
print(f"{final.shape[0]} economías · tick final {df['tick'].max()}")
print(final[['indice', 'activas', 'quiebras', 'fundadas', 'fundadasClub', 'paro', 'clubs', 'socios', 'llamadasMargen', 'corridas']].describe().round(2))
print()
print('mundos con burbuja (índice final ≥ 200):      ', int((final['indice'] >= 200).sum()))
print('mundos con el índice por debajo de la salida:', int((final['indice'] < 100).sum()))
print('mundos con alguna quiebra:                    ', int((final['quiebras'] > 0).sum()))
print('mundos con alguna empresa fundada:            ', int((final['fundadas'] > 0).sum()), ' (por clubes:', int((final['fundadasClub'] > 0).sum()), ')')
print('mundos con paro > 5 % al final:               ', int((final['paro'] > COMPRADORES * 0.05).sum()))
print('mundos con préstamos vivos al final:          ', int((final['prestamosVivos'] > 0).sum()))
print('mundos con llamadas de margen:                ', int((final['llamadasMargen'] > 0).sum()))
print('mundos con crisis de liquidez bancaria:       ', int((final['iliquidez'] > 0).sum()))
print('mundos con corridas contra un club:           ', int((final['corridas'] > 0).sum()))

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(2, 3, figsize=(16, 8))
ax[0, 0].hist(final['indice'], bins=40, color='#2a6f97'); ax[0, 0].set_title('índice de bolsa al final (monedas; salida = 100)')
ax[0, 1].hist(final['quiebras'], bins=range(0, int(final['quiebras'].max()) + 2), color='#c0392b', align='left'); ax[0, 1].set_title('quiebras por mundo')
ax[1, 0].hist(final['fundadas'], bins=range(0, int(final['fundadas'].max()) + 2), color='#27ae60', align='left'); ax[1, 0].set_title('empresas fundadas por mundo')
ax[1, 1].hist(final['paro'] * 100 / COMPRADORES, bins=40, color='#8e44ad'); ax[1, 1].set_title('desempleo al final (%)')
ax[0, 2].hist(final['socios'], bins=40, color='#d35400'); ax[0, 2].set_title('socios de clubes por mundo')
ax[1, 2].hist(final['prestamosVivos'] / 100, bins=40, color='#16a085'); ax[1, 2].set_title('préstamos vivos al final (monedas)')
plt.tight_layout(); plt.show()

# el abanico: los mil índices a lo largo del tiempo, con la mediana y las bandas 5–95 y 25–75
piv = df.pivot(index='tick', columns='economia', values='indice')
h = piv.index / 3600
q = piv.quantile([0.05, 0.25, 0.5, 0.75, 0.95], axis=1).T
plt.figure(figsize=(13, 5))
plt.fill_between(h, q[0.05], q[0.95], color='#2a6f97', alpha=0.2, label='5–95 %')
plt.fill_between(h, q[0.25], q[0.75], color='#2a6f97', alpha=0.4, label='25–75 %')
plt.plot(h, q[0.5], color='#2a6f97', lw=2, label='mediana')
for k in piv.columns[:20]: plt.plot(h, piv[k], color='k', lw=0.3, alpha=0.4)
plt.xlabel('horas simuladas'); plt.ylabel('índice (monedas)'); plt.title(f'{piv.shape[1]} mundos: el abanico del índice de bolsa (20 mundos dibujados)')
plt.legend(); plt.show()

## Qué hacer con esto

- **Más horas**: `TICKS = 28800` (8 h) o `86400` (24 h). Las quiebras empiezan cuando los compradores se quedan sin dinero y las ventas bajan; los clubes tardan horas en juntar los 100.000 para fundar; el crédito solo aparece cuando a alguien le falta dinero (sobregiros, nóminas, fundar contra acciones).
- **Sin bancos**: `CREDITO = 0` corre los mismos mundos (mismas semillas) sin crédito, para comparar.
- **Economías más grandes**: `COMPRADORES = 10000` (tope por los 32 bits del dinero) o `POR_SECTOR = 10` (50 empresas al empezar, hasta 80).
- **Cambiar las reglas**: todo está en `ecosim/ecosim-g.esl`, en Ñ. Cada regla es un núcleo `fn ...(h: entero, ...)` que corre una vez por agente, por empresa, por club, por banco o por economía; se cambia el núcleo, se vuelve a correr y los invariantes avisan si algo deja de cuadrar.
- **Reproducir un mundo**: el mundo `k` del proceso 0 es `semilla SEMILLA + k`; correrlo solo (`1 economía`, esa semilla) da exactamente la misma historia, en GPU, nativo o jit.